In [1]:
!pip install pymupdf pandas scikit-learn spacy joblib

In [2]:
import os
print(os.getcwd())

C:\Users\Shivam\anaconda_projects\2606074e-0c34-48a4-8dc4-cb7fcb2ba196\VCONSTRUCT


In [3]:
import fitz

In [4]:
!pip install pymupdf pandas numpy scikit-learn spacy joblib tqdm pdfplumber pytesseract pillow

In [5]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     -------- ------------------------------- 2.6/12.8 MB 14.9 MB/s eta 0:00:01
     ---------------------- ----------------- 7.1/12.8 MB 17.8 MB/s eta 0:00:01
     ----------------------------------- --- 11.8/12.8 MB 19.6 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 19.3 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [6]:
import os
import re
import fitz
import pandas as pd
import numpy as np
import joblib

from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

import spacy
nlp = spacy.load("en_core_web_sm")

In [7]:
import os
import fitz
import pandas as pd

dataset_path = r"C:\Users\Shivam\Downloads\dataset\dataset"

texts = []
labels = []

def extract_text(pdf_path):
    
    doc = fitz.open(pdf_path)
    text = ""
    
    for page in doc:
        text += page.get_text()
        
    return text


for folder in os.listdir(dataset_path):
    
    if folder == "Data to be Classified and Redacted":
        continue
        
    folder_path = os.path.join(dataset_path, folder)
    
    if os.path.isdir(folder_path):
        
        for file in os.listdir(folder_path):
            
            if file.endswith(".pdf"):
                
                pdf_path = os.path.join(folder_path, file)
                
                text = extract_text(pdf_path)
                
                texts.append(text)
                labels.append(folder)

print("Total training samples:", len(texts))

Total training samples: 294


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words="english"
)

X = vectorizer.fit_transform(texts)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (294, 3000)


In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X, labels)

print("Model training completed")

Model training completed


In [10]:
import joblib

joblib.dump(model, "construction_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Model saved successfully")

Model saved successfully


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(classification_report(y_test, predictions))


                 precision    recall  f1-score   support

  Architectural       0.50      0.80      0.62        15
     Electrical       0.60      0.33      0.43         9
Fire Protection       1.00      0.70      0.82        10
     Mechanical       0.62      0.71      0.67         7
       Plumbing       0.69      0.69      0.69        13
     Structural       1.00      0.40      0.57         5

       accuracy                           0.64        59
      macro avg       0.74      0.61      0.63        59
   weighted avg       0.70      0.64      0.64        59



In [12]:
test_folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Data to be Classified and Redacted"

results = []

for file in os.listdir(test_folder):
    
    if file.lower().endswith(".pdf"):
        
        pdf_path = os.path.join(test_folder, file)
        
        text = extract_text(pdf_path)
        
        X_test = vectorizer.transform([text])
        
        prediction = model.predict(X_test)[0]
        
        confidence = max(model.predict_proba(X_test)[0])
        
        results.append({
            "filename": file,
            "predicted_class": prediction,
            "confidence": confidence
        })

print("Total files classified:", len(results))

Total files classified: 50


In [13]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

email_pattern = r'\S+@\S+'
phone_pattern = r'\+?\d[\d -]{8,}\d'

def find_sensitive(text):
    
    sensitive = []
    
    # find emails
    emails = re.findall(email_pattern, text)
    
    # find phones
    phones = re.findall(phone_pattern, text)
    
    sensitive.extend(emails)
    sensitive.extend(phones)
    
    # detect names and organizations
    doc = nlp(text)
    
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE"]:
            sensitive.append(ent.text)
    
    return list(set(sensitive))

In [14]:
import re

email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
phone_pattern = r'\+?\d[\d\s\-]{8,}\d'

def find_sensitive(text):
    
    sensitive = []
    
    emails = re.findall(email_pattern, text)
    phones = re.findall(phone_pattern, text)
    
    sensitive.extend(emails)
    sensitive.extend(phones)
    
    doc = nlp(text)
    
    for ent in doc.ents:
        if ent.label_ in ["PERSON","ORG"]:
            sensitive.append(ent.text)
    
    return list(set(sensitive))

In [3]:
import os
import fitz
import re

test_folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Data to be Classified and Redacted"

redacted_folder = "redacted_pdfs"
os.makedirs(redacted_folder, exist_ok=True)

email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
phone_pattern = r'\+?\d[\d\s\-]{8,}\d'

for file in os.listdir(test_folder):

    if file.lower().endswith(".pdf"):

        pdf_path = os.path.join(test_folder, file)
        doc = fitz.open(pdf_path)

        for page in doc:

            text = page.get_text()

            emails = re.findall(email_pattern, text)
            phones = re.findall(phone_pattern, text)

            sensitive = emails + phones

            for item in sensitive:

                areas = page.search_for(item)

                for area in areas:
                    page.add_redact_annot(area, fill=(0,0,0))

        for page in doc:
            page.apply_redactions()

        output_path = os.path.join(redacted_folder, file)
        doc.save(output_path)
        doc.close()

        print("Redacted:", file)

Redacted: 10 (2).pdf
Redacted: 10.pdf
Redacted: 103.pdf
Redacted: 107.pdf
Redacted: 12.pdf
Redacted: 124.pdf
Redacted: 141.pdf
Redacted: 19 (2).pdf
Redacted: 19.pdf
Redacted: 23.pdf
Redacted: 25R1 OHT Details 26.03.21-OHT.pdf
Redacted: 26.pdf
Redacted: 30 (2).pdf
Redacted: 30.pdf
Redacted: 30R5 Toilet Details 02.04.21-PLUMBING DETAILS.pdf
Redacted: 38.pdf
Redacted: 4.pdf
Redacted: 40 Security Cabin-Security Cabin.pdf
Redacted: 40.pdf
Redacted: 46.pdf
Redacted: 5.pdf
Redacted: 54.pdf
Redacted: 58.pdf
Redacted: 67.pdf
Redacted: 8.pdf
Redacted: a-Rinker_021.pdf
Redacted: a-Rinker_023.pdf
Redacted: a-Rinker_041.pdf
Redacted: A-sample-drawing-package-1and2family.pdf
Redacted: A1.2 - DEMOLITION PLAN SHEETS.pdf
Redacted: E4.1 - ELECTRICAL LOW VOLTAGE NEW WORK PLANS.pdf
Redacted: electricalplan.pdf
Redacted: fp11.pdf
Redacted: fp6.pdf
Redacted: M-1 Mechanical Plan--.pdf
Redacted: m-Rinker_062.pdf
Redacted: m-Rinker_067.pdf
Redacted: m2.pdf
Redacted: P-NGS-riser-phskc-example-plans.pdf
Redacted

In [1]:
import os
import fitz

dataset_path = r"C:\Users\Shivam\Downloads\dataset\dataset"

texts = []
labels = []

classes = [
    "Architectural",
    "Structural",
    "Mechanical",
    "Plumbing",
    "Electrical",
    "Fire Protection"
]

for label in classes:

    folder = os.path.join(dataset_path, label)

    for file in os.listdir(folder):

        if file.lower().endswith(".pdf"):

            pdf_path = os.path.join(folder, file)

            doc = fitz.open(pdf_path)

            text = ""

            for page in doc:
                text += page.get_text()

            texts.append(text)
            labels.append(label)

print("Total samples:", len(texts))

Total samples: 294


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = vectorizer.fit_transform(texts)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (294, 5000)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [4]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)

model.fit(X_train, y_train)

print("Training completed")

Training completed


In [5]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print(classification_report(y_test, predictions))

Accuracy: 0.711864406779661
                 precision    recall  f1-score   support

  Architectural       0.48      0.79      0.59        14
     Electrical       0.67      0.22      0.33         9
Fire Protection       1.00      1.00      1.00         9
     Mechanical       0.67      0.60      0.63        10
       Plumbing       0.92      0.86      0.89        14
     Structural       1.00      0.67      0.80         3

       accuracy                           0.71        59
      macro avg       0.79      0.69      0.71        59
   weighted avg       0.75      0.71      0.70        59



In [6]:
import re

def detect_sheet_code(text):

    if re.search(r'\bA\d{2,3}\b', text):
        return "Architectural"

    if re.search(r'\bS\d{2,3}\b', text):
        return "Structural"

    if re.search(r'\bM\d{2,3}\b', text):
        return "Mechanical"

    if re.search(r'\bP\d{2,3}\b', text):
        return "Plumbing"

    if re.search(r'\bE\d{2,3}\b', text):
        return "Electrical"

    if re.search(r'\bFP\d{2,3}\b', text):
        return "Fire Protection"

    return None

In [7]:
code = detect_sheet_code(text)

if code:
    prediction = code
else:
    prediction = model.predict(X_test)[0]

predictions = model.predict(X_test)

In [8]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.711864406779661


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=8000,
    ngram_range=(1,2)   # unigram + bigram
)

X = vectorizer.fit_transform(texts)

print(X.shape)

(294, 8000)


In [10]:
def keyword_features(text):

    keywords = {
        "Architectural": ["floor","elevation","door","window"],
        "Structural": ["beam","column","rebar"],
        "Mechanical": ["duct","hvac","ventilation"],
        "Plumbing": ["pipe","drain","fixture","plumbed"],
        "Electrical": ["panel","lighting","wiring"],
        "Fire Protection": ["sprinkler","alarm","hydrant"]
    }

    counts = []

    text = text.lower()

    for cls in keywords:
        count = sum(text.count(word) for word in keywords[cls])
        counts.append(count)

    return counts

In [11]:
import numpy as np

keyword_matrix = np.array([keyword_features(t) for t in texts])

In [12]:
from scipy.sparse import hstack

X_combined = hstack([X, keyword_matrix])

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [15]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("New Accuracy:", accuracy)

print(classification_report(y_test, predictions))

New Accuracy: 0.711864406779661
                 precision    recall  f1-score   support

  Architectural       0.50      0.79      0.61        14
     Electrical       0.67      0.22      0.33         9
Fire Protection       1.00      1.00      1.00         9
     Mechanical       0.67      0.60      0.63        10
       Plumbing       0.86      0.86      0.86        14
     Structural       1.00      0.67      0.80         3

       accuracy                           0.71        59
      macro avg       0.78      0.69      0.71        59
   weighted avg       0.74      0.71      0.70        59



In [16]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X,
    labels,
    cv=cv,
    scoring="accuracy"
)

print("Scores:", scores)
print("Average accuracy:", scores.mean())

Scores: [0.71186441 0.71186441 0.62711864 0.72881356 0.75862069]
Average accuracy: 0.707656341320865


In [18]:
from sklearn.linear_model import LogisticRegression
final_model = LogisticRegression(max_iter=20000)

final_model.fit(X, labels)

print("Final model trained on full dataset")

Final model trained on full dataset


In [19]:
import joblib

joblib.dump(final_model, "construction_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model saved successfully")

Model saved successfully


In [20]:
import fitz   # PyMuPDF

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    
    text = ""
    
    for page in doc:
        text += page.get_text()
        
    doc.close()
    
    return text

In [21]:
import os

folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Architectural"

files = os.listdir(folder)

pdf_file = [f for f in files if f.lower().endswith(".pdf")][0]

sample_pdf = os.path.join(folder, pdf_file)

print("Using file:", sample_pdf)

text = extract_text(sample_pdf)

print(text[:500])


Using file: C:\Users\Shivam\Downloads\dataset\dataset\Architectural\1.pdf
WOOD PLANK BENCH
WOOD PLANK BENCH
SOFA
SINK
LOCKER
MODULAR SINGLE BED
W/ DESK
TELEVISION
WC.
F.D.
DESK
ISOLATION RM 3. w/ T&B
AREA: 12.0sq.mtrs.
T&B
ACU
FLOATING SHELF
SOFA
SINK
LOCKER
MODULAR SINGLE BED
W/ DESK
TELEVISION
SHO.
WC.
F.D.
DESK
ISOLATION RM. 2 w/ T&B
AREA: 12.0sq.mtrs.
T&B
ACU
FLOATING SHELF
SOFA
SINK
LOCKER
MODULAR SINGLE BED
W/ DESK
TELEVISION
WC.
F.D.
DESK
ISOLATION RM. 1 w/ T&B
AREA: 12.0sq.mtrs.
T&B
ACU
FLOATING SHELF
NICHE
NICHE
NICHE
SHO.
SHO.
WOOD PLANK BENCH
VERANDA
EL. + 


In [22]:
model.fit(X, labels)


RandomForestClassifier(n_estimators=300, random_state=42)

In [30]:
import os
import fitz
import pandas as pd
import joblib

# -----------------------------
# Load trained model and vectorizer
# -----------------------------

model = joblib.load("construction_classifier.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# -----------------------------
# Path to test data
# -----------------------------

test_folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Data to be Classified and Redacted"

# -----------------------------
# Function to extract text
# -----------------------------

def extract_text(pdf_path):
    
    doc = fitz.open(pdf_path)
    text = ""
    
    for page in doc:
        text += page.get_text()
        
    doc.close()
    
    return text

# -----------------------------
# Run predictions
# -----------------------------

results = []

for file in os.listdir(test_folder):
    
    if file.lower().endswith(".pdf"):
        
        pdf_path = os.path.join(test_folder, file)
        
        text = extract_text(pdf_path)
        
        X_test = vectorizer.transform([text])
        
        prediction = model.predict(X_test)[0]
        
        confidence = model.predict_proba(X_test).max()
        
        results.append({
            "filename": file,
            "predicted_class": prediction,
            "confidence": confidence
        })

# -----------------------------
# Save results to CSV
# -----------------------------

df = pd.DataFrame(results)

df.to_csv("classification_output.csv", index=False)

print("Classification completed")
print(df.head(20))

Classification completed
                                             filename predicted_class  \
0                                          10 (2).pdf   Architectural   
1                                              10.pdf      Mechanical   
2                                             103.pdf      Electrical   
3                                             107.pdf      Electrical   
4                                              12.pdf   Architectural   
5                                             124.pdf   Architectural   
6                                             141.pdf   Architectural   
7                                          19 (2).pdf        Plumbing   
8                                              19.pdf      Mechanical   
9                                              23.pdf      Mechanical   
10                  25R1 OHT Details 26.03.21-OHT.pdf   Architectural   
11                                             26.pdf        Plumbing   
12                        